In [1]:
import numpy as np
import matplotlib.pyplot as plt
import re
import json
from functions import *
import pandas as pd
import csv

def get_currents(file):
    def read_currents_file(filepath):
        # Read the file content
        with open(filepath, 'r') as file:
            # Read the first line to get current names
            header_line = file.readline().strip()
            
            # Split the header first by semicolons, then by commas
            current_groups = header_line.split(';')
            current_names = []
            for group in current_groups:
                currents = [name.strip() for name in group.split(',')]
                current_names.extend(currents)
                
            # Initialize dictionary with empty lists for each current
            data_dict = {name: [] for name in current_names}
            
            # Read the rest of the lines
            for line in file:
                # if start with undefined,skip
                if line.startswith("undefined"):
                    continue
                if not line.strip():  # Skip empty lines
                    continue
                
                # Split values by semicolon first, then comma
                value_groups = line.strip().split(';')
                values = []
                for group in value_groups:
                    group_values = [float(val.strip()) for val in group.split(',') if val.strip()]
                    values.extend(group_values)
                
                # Add each value to corresponding current's list
                for name, value in zip(current_names, values):
                    data_dict[name].append(value)
        
        # Convert lists to numpy arrays for easier manipulation
        for name in data_dict:
            data_dict[name] = np.array(data_dict[name])
        
        return data_dict

    # Example usage:
    filepath = file
    currents_data = read_currents_file(filepath)
    return currents_data
def readFile(file,unidentifiable_space = [3,5,6,7,8,9,10,11]):
    if not unidentifiable_space:
        unidentifiable_space = list(range(12))
    currents_data = get_currents(file)
    mask =  -100< currents_data['voltage']
    if 'NA' in currents_data:
        del currents_data['NA']
    matrix = np.zeros((len(currents_data['voltage'][mask]),len(currents_data)-1))  # Exclude voltage
    # assign currents to matrix columns
    for i, (current_name, values) in enumerate(currents_data.items()):
        if current_name not in ['voltage']:
            matrix[:, i - (1 if 'voltage' in currents_data else 0)] = values[mask]
    U, S, V = np.linalg.svd(matrix)
    def projection_S(v):
        ps = [0] * len(S)
        for i in unidentifiable_space:
            ps += np.inner(v,V[i]) / np.inner(V[i],V[i]) * V[i]
        return ps
    identifiability = {}
    for i, (current_name, values) in enumerate(currents_data.items()):
        if current_name in ['voltage']:
            continue
        v_I = [0]*len(S)
        v_I[i-1] = 1
        k = np.linalg.norm(v_I-projection_S(v_I))
        identifiability[current_name] = k
    sorted_identifiability = dict(sorted(identifiability.items(), key=lambda item: item[1], reverse=True))
    return U,S,V,currents_data,sorted_identifiability
def plot_sorted_currents_with_identifiability(currents_data, sorted_identifiability):
    # Get currents in sorted order (already sorted by identifiability)
    currents_to_plot = list(sorted_identifiability.keys())
    n_currents = len(currents_to_plot)
    
    # Create figure with special grid
    fig = plt.figure(figsize=(15, 4*(n_currents//3 + 2)))  # +2 for voltage row
    
    # Create grid with different row heights
    gs = plt.GridSpec(n_currents//3 + 2, 3, height_ratios=[1.5] + [1]*(n_currents//3 + 1))
    
    # Plot voltage across entire first row
    ax_voltage = fig.add_subplot(gs[0, :])
    ax_voltage.plot(currents_data['voltage'], 'b-')
    ax_voltage.set_title('Voltage')
    ax_voltage.set_ylabel('mV')
    ax_voltage.set_xlabel('Time Step')
    ax_voltage.grid(True)
    
    # Plot currents in remaining grid
    for idx, current_name in enumerate(currents_to_plot):
        row = (idx // 3) + 1  # +1 because voltage took first row
        col = idx % 3
        ax = fig.add_subplot(gs[row, col])
        
        # Plot the current
        ax.plot(currents_data[current_name], 'b-')
        
        # Set title with identifiability value
        identifiability_value = sorted_identifiability[current_name]
        def format_current_name(name):
            # Dictionary for special current name formatting
            current_formats = {
                'INa': 'I_{Na}',
                'ICaL': 'I_{CaL}',
                'Ito': 'I_{to}',
                'IKr': 'I_{Kr}',
                'IKs': 'I_{Ks}',
                'IK1': 'I_{K1}',
                'INaCa': 'I_{NaCa}',
                'INaK': 'I_{NaK}',
                'INab': 'I_{Nab}',
                'ICab': 'I_{Cab}',
                'IKb': 'I_{Kb}',
                'IpCa': 'I_{pCa}',
                'INalate': 'I_{Na,late}'
            }
            return current_formats.get(name, name)  # Return formatted name or original if not in dictionary

        # Then modify the title setting line to:
        ax.set_title(f'${format_current_name(current_name)}$\nIdentifiability: {identifiability_value:.3f}',fontsize=20)        
        ax.set_ylabel('Current (pA/pF)')
        ax.set_xlabel('Time Step')
        ax.grid(True)
    
    #plt.tight_layout()
    #plt.savefig('currents_sorted_by_identifiability.png', dpi=300)
    #plt.show()
    plt.close()
def get_voltage(vec,epsilon,pacing_period,drug_name):
    path = f"./data-5xdrug/voltage_vector{vec}_epsilon_{epsilon}_pacingPeriod_{pacing_period}_drug_name_{drug_name}.csv"
    voltage = np.array([])
    with open(path) as csv_file:
        csv_reader = csv.reader(csv_file, delimiter=';')
        for row in csv_reader:
            voltage = np.append(voltage, float(row[1]))
    return voltage

def dvdt_max(voltage):
    return np.max(np.gradient(voltage)) / dt
def euclidian_norm(v):
    return np.linalg.norm(v)

def H_test(v_star,v_bar):
    H1 = abs(get_APD_from_voltage(v_star,0.7,dt)[0] - get_APD_from_voltage(v_bar,0.7,dt)[0]) / abs(get_APD_from_voltage(v_star,0.7,dt)[0])
    H2 = abs(get_APD_from_voltage(v_star,0.5,dt)[0] - get_APD_from_voltage(v_bar,0.5,dt)[0]) / abs(get_APD_from_voltage(v_star,0.5,dt)[0])       
    H3 = abs(get_APD_from_voltage(v_star,0.2,dt)[0] - get_APD_from_voltage(v_bar,0.2,dt)[0]) / abs(get_APD_from_voltage(v_star,0.2,dt)[0])
    H4 = abs(dvdt_max(v_star) - dvdt_max(v_bar)) / abs(dvdt_max(v_star))
    H5 = np.linalg.norm(v_star - v_bar) / np.linalg.norm(v_star)
    return H1 + H2 + H3 + H4 + H5
    
def calculate_unidentifiable_space(pacing_period,drug_name):
    voltage_file = f'2D-TNNP-pacing-period-{pacing_period}-5xdrug/voltage_TNNP_pacingPeriod_{pacing_period}_{drug_name}.csv'
    vec_lst = list(range(12))
    epsilon_list = [-0.5,0.5]
    Delta = 0.25
    output = list(range(12))
    for vec in vec_lst:
        for epsilon in epsilon_list:            
            v_star = get_currents(voltage_file)['voltage']
            v_bar = get_voltage(vec,epsilon,pacing_period,drug_name)
            H_value = H_test(v_star,v_bar[1:])
            if H_value > Delta:
                # remove vec from output
                output.remove(vec)
                break
    return output

In [2]:
#calculate unidentifiable space
dt = 0.2

In [3]:
def get_IIC(drug_name,pacing_period):   
    file = f'2D-TNNP-pacing-period-{pacing_period}-5xdrug/voltage_TNNP_pacingPeriod_{pacing_period}_{drug_name}.csv'
    unidentifiable_space = calculate_unidentifiable_space(pacing_period,drug_name)
    print(f"Unidentifiable space for drug {drug_name} at pacing period {pacing_period}: {unidentifiable_space}")
    U,S,V,currents_data, sorted_identifiability = readFile(file,unidentifiable_space)
    return sorted_identifiability
    

In [4]:
pacing_period = 1000
folder = f"2D-TNNP-pacing-period-{pacing_period}-5xdrug"
# and find the pkl file
pkl_files = [f for f in os.listdir(folder) if f.endswith('.pkl')]

for file in pkl_files:
    with open(os.path.join(folder, file), 'rb') as f:
        drug_dict = pickle.load(f)

In [5]:
IIC_dict = {}
for drug_name in drug_dict.keys():
    if 'INaL' in drug_dict[drug_name]:
        print(f"Skipping simulation for {drug_name} due to INaL involvement...")
        continue
    sorted_identifiability = get_IIC(drug_name,pacing_period)
    IIC_dict[drug_name] = sorted_identifiability

# save IIC_dict as pkl file to certain folder
with open(os.path.join('data-5xdrug', f'IIC_dict_pacing_period_{pacing_period}.pkl'), 'wb') as f:
    pickle.dump(IIC_dict, f)

Unidentifiable space for drug Amiodarone I at pacing period 1000: [3, 5, 6, 7, 8, 9, 10, 11]
Skipping simulation for Amiodarone II due to INaL involvement...
Unidentifiable space for drug Astemizole at pacing period 1000: [3, 5, 6, 7, 8, 9, 10, 11]
Unidentifiable space for drug BaCl2 at pacing period 1000: [3, 5, 6, 7, 8, 9, 10, 11]
Unidentifiable space for drug Bepridil I at pacing period 1000: [3, 5, 6, 7, 8, 9, 10, 11]
Unidentifiable space for drug Bepridil II at pacing period 1000: [3, 5, 6, 7, 8, 9, 10, 11]
Skipping simulation for Bepridil III due to INaL involvement...
Unidentifiable space for drug Ceftriaxone at pacing period 1000: [5, 6, 7, 8, 9, 10, 11]
Unidentifiable space for drug Chloropromazine I at pacing period 1000: [3, 5, 6, 7, 8, 9, 10, 11]
Skipping simulation for Chloropromazine II due to INaL involvement...
Unidentifiable space for drug Cilostazol at pacing period 1000: [3, 5, 6, 7, 8, 9, 10, 11]
Unidentifiable space for drug Cisapride I at pacing period 1000: [3, 5

In [10]:
# print IIC_dict, all details and save digits after decimal point to 3
for drug_name, identifiability in IIC_dict.items():
    print(f"Drug: {drug_name}")
    for current, value in identifiability.items():
        print(f"  {current}: {value:.3f}")

Drug: Amiodarone I
  INa: 1.000
  Ito: 0.963
  IKs: 0.892
  ICaL: 0.882
  INaK: 0.406
  IpK: 0.390
  IKr: 0.302
  INaCa: 0.220
  IK1: 0.170
  IbCa: 0.098
  IpCa: 0.059
  IbNa: 0.033
Drug: Astemizole
  INa: 1.000
  Ito: 0.953
  ICaL: 0.873
  IKs: 0.862
  IpK: 0.631
  INaK: 0.309
  IKr: 0.214
  INaCa: 0.166
  IK1: 0.114
  IbCa: 0.058
  IpCa: 0.057
  IbNa: 0.019
Drug: BaCl2
  INa: 1.000
  Ito: 0.940
  ICaL: 0.870
  IKs: 0.831
  IpK: 0.614
  INaK: 0.323
  IKr: 0.309
  INaCa: 0.242
  IK1: 0.155
  IbCa: 0.074
  IpCa: 0.061
  IbNa: 0.026
Drug: Bepridil I
  INa: 1.000
  Ito: 0.954
  IKs: 0.894
  ICaL: 0.870
  IpK: 0.610
  INaK: 0.331
  INaCa: 0.144
  IK1: 0.120
  IKr: 0.106
  IbCa: 0.063
  IpCa: 0.058
  IbNa: 0.020
Drug: Bepridil II
  INa: 1.000
  Ito: 0.956
  IKs: 0.883
  ICaL: 0.874
  IpK: 0.603
  INaK: 0.329
  IKr: 0.167
  INaCa: 0.150
  IK1: 0.118
  IbCa: 0.063
  IpCa: 0.058
  IbNa: 0.021
Drug: Ceftriaxone
  INa: 1.000
  Ito: 0.987
  ICaL: 0.926
  IK1: 0.757
  IKs: 0.690
  INaCa: 0.639
  I